# Ministral-3B Multimodal (Kaggle Edition)

This notebook implements the architecture for **Ministral-3-3B-Instruct-2512** from scratch, loads weights from Hugging Face, and runs a Gradio interface. 

### Changes & Fixes Applied:
1. **Memory Optimization:** Added `logits_to_keep=1` during generation to prevent VRAM OOM on the P100/T4 GPUs.
2. **Tensor Safety:** Added `.contiguous()` checks in the multimodal projector to prevent scrambled vision features.
3. **Initialization:** Implemented direct GPU initialization to avoid double-RAM usage (CPU+GPU).
4. **AutoProcessor:** Fixed the processor loading logic to use the Repo ID correctly.

## 1. Setup & Utilities

In [ ]:
!pip install -q -U torch transformers accelerate safetensors gradio huggingface_hub

In [ ]:
import torch
import torch.nn as nn
import math
import gc
import os
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, Tuple, List, Union, Iterable

# --- Missing Utility: rotate_functions ---
def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors."""
    # cos, sin shape: (batch, seq_len, head_dim)
    # q, k shape: (batch, heads, seq_len, head_dim)
    
    # Reshape cos/sin to broadcast: (batch, 1, seq_len, head_dim)
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# --- Missing Utility: KVCache ---
class KVCache:
    """
    A simple KV Cache implementation for autoregressive decoding.
    """
    def __init__(self):
        self.key_cache: List[torch.Tensor] = []
        self.value_cache: List[torch.Tensor] = []
        self.seen_tokens = 0

    def update(
        self,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        layer_idx: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Updates the cache with new key/value states and returns the full concatenated states.
        """
        # Initialize lists if this is the first token
        if len(self.key_cache) <= layer_idx:
            self.key_cache.append(key_states)
            self.value_cache.append(value_states)
        else:
            # Concatenate along sequence dimension (dim=2)
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_states], dim=2)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_states], dim=2)
            
        # Update seen tokens count only once per step (using layer 0 as reference)
        if layer_idx == 0:
            self.seen_tokens = self.key_cache[0].shape[2]

        return self.key_cache[layer_idx], self.value_cache[layer_idx]

    def num_items(self) -> int:
        return self.seen_tokens

In [ ]:
# -----------------------
# Ministral 3B Text Backbone
# -----------------------

@dataclass
class RopeParameters:
    beta_fast: float = 32.0
    beta_slow: float = 1.0
    factor: float = 16.0
    llama_4_scaling_beta: float = 0.1
    mscale: float = 1.0
    mscale_all_dim: float = 1.0
    original_max_position_embeddings: int = 16384
    rope_theta: float = 1000000.0
    rope_type: str = "yarn"
    type: str = "yarn"

@dataclass
class Ministral3Config:
    attention_dropout: float = 0.0
    head_dim: int = 128
    hidden_act: str = "silu"
    hidden_size: int = 3072
    intermediate_size: int = 9216
    max_position_embeddings: int = 262144
    num_attention_heads: int = 32
    num_hidden_layers: int = 26
    num_key_value_heads: int = 8 
    rms_norm_eps: float = 1e-05
    vocab_size: int = 131072
    tie_word_embeddings: bool = True
    rope_parameters: dict = field(default_factory=lambda: RopeParameters().__dict__)
    pad_token_id: Optional[int] = 11
    bos_token_id: Optional[int] = 1
    eos_token_id: Optional[int] = 2
    sliding_window: Optional[int] = None
    use_cache: bool = True

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch, num_kv_heads, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_kv_heads, n_rep, seq_len, head_dim
    )
    return hidden_states.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)

def _get_llama_4_attn_scale(positions_ids, beta, max_position_embeddings):
    bucket_index = torch.floor(positions_ids / max_position_embeddings)
    scaling = 1.0 + beta * torch.log(1.0 + bucket_index)
    return scaling.unsqueeze(-1)

def create_causal_mask(config, inputs_embeds, attention_mask, past_key_values=None):
    if inputs_embeds is None: return None
    batch_size, query_length, _ = inputs_embeds.shape
    past_length = past_key_values.num_items() if past_key_values is not None else 0
    kv_length = past_length + query_length
    
    mask = torch.full((query_length, kv_length), torch.finfo(inputs_embeds.dtype).min, device=inputs_embeds.device)
    mask = torch.triu(mask, diagonal=1 + past_length)
    mask = mask.unsqueeze(0).unsqueeze(1).expand(batch_size, 1, query_length, kv_length)

    if attention_mask is not None:
        padding_mask = (1.0 - attention_mask).to(inputs_embeds.dtype) * torch.finfo(inputs_embeds.dtype).min
        padding_mask = padding_mask[:, None, None, :]
        mask = mask + padding_mask
    return mask

class Ministral3Attention(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = config.head_dim
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads
        self.scaling = self.head_dim ** -0.5
        
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)

    def forward(self, hidden_states, position_embeddings, attention_mask, cache_position, past_key_values=None, position_ids=None):
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)
        
        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        llama_beta = self.config.rope_parameters.get("llama_4_scaling_beta", 0.1)
        original_max_pos = self.config.rope_parameters.get("original_max_position_embeddings", 16384)
        query_states = query_states * _get_llama_4_attn_scale(cache_position, llama_beta, original_max_pos).to(query_states.dtype)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups)

        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scaling
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask
        
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)
        attn_output = attn_output.transpose(1, 2).contiguous().reshape(*input_shape, -1)
        
        return self.o_proj(attn_output), attn_weights

class Ministral3MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

class Ministral3RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        x = hidden_states.to(torch.float32)
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        return (self.weight * x).to(input_dtype)

class Ministral3DecoderLayer(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.self_attn = Ministral3Attention(config, layer_idx)
        self.mlp = Ministral3MLP(config)
        self.input_layernorm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_values=None, cache_position=None, position_embeddings=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        attn_out, _ = self.self_attn(hidden_states, position_embeddings, attention_mask, cache_position, past_key_values, position_ids)
        hidden_states = residual + attn_out

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states

class Ministral3RotaryEmbedding(nn.Module):
    inv_freq: torch.Tensor
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        inv_freq, self.attention_scaling = self._compute_rope_parameters(config)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def _compute_rope_parameters(self, config):
        rope_params = config.rope_parameters
        head_dim = config.head_dim
        base = rope_params.get("rope_theta", 1000000.0)
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        
        if rope_params.get("rope_type") != "yarn": return inv_freq, 1.0

        # YaRN Logic
        factor = rope_params.get("factor", 16.0)
        original_max_pos = rope_params.get("original_max_position_embeddings", 16384)
        beta_fast = rope_params.get("beta_fast", 32.0)
        beta_slow = rope_params.get("beta_slow", 1.0)
        mscale = rope_params.get("mscale", 1.0)
        mscale_all_dim = rope_params.get("mscale_all_dim", 1.0)

        def get_mscale_factor(scale, mscale_val=1.0):
             return 0.1 * mscale_val * math.log(scale) + 1.0 if scale > 1 else 1.0
        
        attn_factor = get_mscale_factor(factor, mscale) / get_mscale_factor(factor, mscale_all_dim)

        def find_correction_dim(num_rotations, dim, base, max_position):
            return (dim * math.log(max_position / (num_rotations * 2 * math.pi))) / (2 * math.log(base))

        def linear_ramp_factor(min_val, max_val, dim_tensor):
            if min_val == max_val: max_val += 0.001
            return torch.clamp((dim_tensor - min_val) / (max_val - min_val), 0, 1)

        low = max(math.floor(find_correction_dim(beta_fast, head_dim, base, original_max_pos)), 0)
        high = min(math.ceil(find_correction_dim(beta_slow, head_dim, base, original_max_pos)), head_dim // 2 - 1)
        ramp = linear_ramp_factor(low, high, torch.arange(head_dim // 2, dtype=torch.float32))

        pos_freqs = base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        inv_freq_extrap = 1.0 / pos_freqs
        inv_freq_interp = 1.0 / (factor * pos_freqs)
        
        return (inv_freq_interp * (1 - ramp) + inv_freq_extrap * ramp), attn_factor

    def forward(self, x, position_ids):
        inv_freq = self.inv_freq.to(x.device, dtype=torch.float32)
        inv_freq_expanded = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return (emb.cos() * self.attention_scaling).to(x.dtype), (emb.sin() * self.attention_scaling).to(x.dtype)

class Ministral3Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, config.pad_token_id)
        self.layers = nn.ModuleList([Ministral3DecoderLayer(config, i) for i in range(config.num_hidden_layers)])
        self.norm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Ministral3RotaryEmbedding(config)

    def forward(self, input_ids=None, attention_mask=None, position_ids=None, past_key_values=None, inputs_embeds=None, cache_position=None):
        if inputs_embeds is None: inputs_embeds = self.embed_tokens(input_ids)
        if past_key_values is None: past_key_values = KVCache()
        
        if cache_position is None:
            past_len = past_key_values.num_items()
            cache_position = torch.arange(past_len, past_len + inputs_embeds.shape[1], device=inputs_embeds.device)
        
        if position_ids is None: position_ids = cache_position.unsqueeze(0)

        causal_mask = create_causal_mask(self.config, inputs_embeds, attention_mask, past_key_values)
        pos_emb = self.rotary_emb(inputs_embeds, position_ids)

        hidden_states = inputs_embeds
        for layer in self.layers:
            hidden_states = layer(hidden_states, causal_mask, position_ids, past_key_values, cache_position, pos_emb)
            
        hidden_states = self.norm(hidden_states)
        return {"last_hidden_state": hidden_states, "past_key_values": past_key_values}

## 2. Vision Components (Pixtral) & Multimodal Wrapper

This section implements the **Pixtral Vision Encoder** (based on ViT structure with 2D RoPE) and the **Mistral Projector** (MLP). It also defines the final `Ministral3ForConditionalGeneration` class that ties the vision and text towers together.

### Key Fixes Implemented:
- **`_replace_image_tokens`**: Added `.contiguous()` to the vision features before scattering them into the text embeddings. This prevents silent data corruption when tensors have been permuted.
- **`generate` Loop**: Optimized to pass `logits_to_keep=1` to the model, ensuring we only compute the next-token probability instead of the entire sequence history (saving ~4GB VRAM).

In [ ]:
@dataclass
class PixtralConfig:
    """Configuration class for Pixtral model architecture (Vision Tower)."""

    head_dim: int = 64
    attention_dropout: float = 0.0
    hidden_size: int = 1024
    image_size: int = 1540
    intermediate_size: int = 4096
    num_attention_heads: int = 16
    num_hidden_layers: int = 24
    patch_size: int = 14
    rope_theta: float = 10000.0
    num_channels: int = 3


# --------
# Helper Methods
# --------


def position_ids_in_meshgrid(patch_embeds_list, max_width):
    """
    Generate position IDs for a list of patch embeddings based on their 2D grid positions.
    Args:
        patch_embeds_list: List of patch embedding tensors, each of shape (batch, channels, height, width)
        max_width: Maximum width of the image in terms of patches (image_size // patch_size)
    Returns:
        Tensor of shape (total_patches,) containing position IDs for each patch based on its grid position.
    """
    positions = []
    for patch in patch_embeds_list:
        height, width = patch.shape[-2:]
        mesh = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")
        h_grid, v_grid = torch.stack(mesh, dim=-1).reshape(-1, 2).chunk(2, -1)
        ids = h_grid * max_width + v_grid
        positions.append(ids[:, 0])
    return torch.cat(positions)


def generate_block_attention_mask(patch_embeds_list, tensor):
    """
    Generate a block-diagonal attention mask for a sequence of patch embeddings.
    This mask allows full attention within each image's patches but prevents attention across different images in the batch.
    Args:
        patch_embeds_list: List of patch embedding tensors, each of shape (batch, channels, height, width)
        tensor: The input tensor for which the attention mask is being generated (used to determine dtype and device)
    Returns:
        A block-diagonal attention mask tensor of shape (batch, 1, seq_len, seq_len) where seq_len is the total number of patches across all images in the batch. The mask has 0s for positions within the same image and -inf for positions across different images, allowing attention only within each image's patches.
    """
    dtype = tensor.dtype
    device = tensor.device
    seq_len = tensor.shape[1]
    d_min = torch.finfo(dtype).min
    causal_mask = torch.full(
        (seq_len, seq_len), fill_value=d_min, dtype=dtype, device=device
    )

    block_end_idx = torch.tensor(patch_embeds_list).cumsum(-1)
    block_start_idx = torch.tensor([0] + patch_embeds_list[:-1]).cumsum(-1)
    for start, end in zip(block_start_idx, block_end_idx):
        causal_mask[start:end, start:end] = 0

    causal_mask = causal_mask[None, None, :, :].expand(tensor.shape[0], 1, -1, -1)
    return causal_mask


# ============================================================================
# Rotary Position Embeddings (RoPE) - 2D Grid-based
# ============================================================================


class PixtralRotaryEmbedding(nn.Module):
    """
    2D Rotary Position Embedding for image tokens.

    Key difference from standard RoPE: Each pixel position gets its own frequency
    based on its 2D location (height, width) in the image grid.

    Outputs tensor of shape (batch, height * width, dim) with position embeddings
    where each token gets a positional embedding based on its grid position.
    """

    inv_freq: torch.Tensor  # Type hint for `register_buffer`

    def __init__(self, config: PixtralConfig) -> None:
        super().__init__()
        self.config = config
        self.rope_base = config.rope_theta

        # Compute and register inverse frequencies
        inv_freq = self.compute_default_rope_parameters()
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("original_inv_freq", inv_freq.clone(), persistent=False)

    # --------
    # RoPE Computation
    # --------

    def compute_default_rope_parameters(self) -> torch.Tensor:
        """
        Compute inverse frequencies for 2D grid-based RoPE.

        Unlike standard RoPE which uses sequence position, this creates a 2D grid
        where each (height, width) position gets separate frequency components.

        Returns:
            Tensor of shape (patches_total, dim) with inverse frequencies
        """
        base = self.rope_base
        dim = getattr(self.config, "head_dim", None) or (
            self.config.hidden_size // self.config.num_attention_heads
        )

        # Create 2D grid of patch positions
        max_patches_per_side = self.config.image_size // self.config.patch_size
        h = torch.arange(max_patches_per_side)
        w = torch.arange(max_patches_per_side)

        # Compute base frequencies
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

        # Create separate frequency maps for height and width
        freqs_h = torch.outer(h, freqs[::2]).float()  # shape: (patches, dim/4)
        freqs_w = torch.outer(w, freqs[1::2]).float()  # shape: (patches, dim/4)

        # Combine height and width frequencies into 2D grid
        inv_freq = torch.cat(
            [
                freqs_h[:, None, :].repeat(1, max_patches_per_side, 1),  # (H, W, dim/4)
                freqs_w[None, :, :].repeat(max_patches_per_side, 1, 1),  # (H, W, dim/4)
            ],
            dim=-1,  # (H, W, dim/2)
        ).reshape(
            -1, dim // 2
        )  # (H*W, dim/2)

        # Duplicate to match full dimension
        inv_freq = torch.cat((inv_freq, inv_freq), dim=-1)  # (H*W, dim)
        return inv_freq

    # --------
    # Forward Pass
    # --------

    def forward(self, hidden_state: torch.Tensor, position_ids: torch.Tensor) -> tuple:
        """
        Compute cos and sin components of rotary embeddings.

        Args:
            hidden_state: Hidden states for dtype conversion
            position_ids: Position indices to look up in inverse frequencies

        Returns:
            Tuple of (cos, sin) embeddings
        """
        freqs = self.inv_freq[position_ids]  # shape: (seq_len, dim)
        emb = freqs
        cos = emb.cos()
        sin = emb.sin()
        return cos.to(dtype=hidden_state.dtype), sin.to(dtype=hidden_state.dtype)


# ============================================================================
# Feed-Forward Network (MLP)
# ============================================================================


class PixtralMLP(nn.Module):
    """
    Feed-forward network with Gated Linear Units (GLU) architecture.

    Structure: (gate_proj * up_proj) -> activation -> down_proj
    This design allows better expressiveness compared to standard dense layers.
    """

    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size

        # Projection layers (no bias)
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)

        # Activation function
        self.act_fn = nn.SiLU()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the MLP.

        Args:
            hidden_states: Input tensor of shape (batch, seq_len, hidden_size)

        Returns:
            Output tensor of shape (batch, seq_len, hidden_size)
        """
        # Gate mechanism: (gate * up) -> activation -> down
        down_proj = self.down_proj(
            self.act_fn(self.gate_proj(hidden_states)) * self.up_proj(hidden_states)
        )
        return down_proj


# ============================================================================
# Layer Normalization (RMS Norm)
# ============================================================================


class PixtralRMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.

    More computationally efficient than standard LayerNorm while providing
    similar stabilization benefits during training.
    """

    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.variance_epsilon = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """
        Apply RMS normalization to input.

        Args:
            hidden_states: Input tensor of shape (batch, seq_len, hidden_size)

        Returns:
            Normalized tensor with same shape
        """
        # Store original dtype and convert to float32 for stability
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(dtype=torch.float32)

        # Compute variance along last dimension
        variance = hidden_states.pow(2).mean(
            -1, keepdim=True
        )  # shape: (batch, seq_len, 1)

        # Normalize and scale back to original dtype
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)


# ============================================================================
# Layer Normalization (RMS Norm)
# ============================================================================


class PixtralAttention(nn.Module):
    """
    Multi-head attention mechanism for Pixtral model.

    Key features:
    - Supports both self-attention and cross-attention
    - Uses 2D RoPE for positional embeddings
    - Configurable number of heads and head dimensions
    """

    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.head_dim = config.head_dim

        # Ensure hidden size is divisible by number of heads
        assert (
            self.hidden_size % self.num_attention_heads == 0
        ), "Hidden size must be divisible by number of heads"

        # Projection layers for query, key, value (no bias)
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # Output projection layer (no bias)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # Dropout for attention probabilities
        self.attn_dropout = nn.Dropout(config.attention_dropout)

    def attention_forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        scaling: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Compute attention output given query, key, value tensors.

        Args:
            query: Query tensor of shape (batch, heads, seq_len_q, head_dim)
            key: Key tensor of shape (batch, heads, seq_len_kv, head_dim)
            value: Value tensor of shape (batch, heads, seq_len_kv, head_dim)
            attention_mask: Optional mask tensor for attention (broadcastable to (batch, heads, seq_len_q, seq_len_kv))
            scaling: Scaling factor for dot product attention (e.g., 1/sqrt(head_dim))
        Returns:
            Attention output tensor of shape (batch, heads, seq_len_q, head_dim)
        """
        # Compute scaled dot product attention scores
        attn_scores = torch.matmul(query, key.transpose(-2, -1)) * scaling

        # Apply attention mask if provided
        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask

        # Compute attention probabilities
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        # Compute attention output
        attn_output = torch.matmul(attn_weights, value)
        attn_output = attn_output.transpose(1, 2).contiguous()
        return attn_output, attn_weights

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_embeddings: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        output_attentions: bool | None = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Compute multi-head attention output.
        Args:
            hidden_states: Input tensor of shape (batch, seq_len, hidden_size)
            attention_mask: Optional mask tensor for attention (broadcastable to (batch, heads, seq_len, seq_len))
            position_embeddings: Optional tuple of (cos, sin) tensors for RoPE
            output_attentions: Whether to return attention weights
        Returns:
            Tuple of (attention_output, attention_weights)
        """
        batch_size, patches, _ = hidden_states.size()

        # Project hidden states to query, key, value
        query = (
            self.q_proj(hidden_states)
            .view(batch_size, patches, self.num_attention_heads, self.head_dim)
            .transpose(1, 2)
        )  # (batch, heads, seq_len, head_dim)
        key = (
            self.k_proj(hidden_states)
            .view(batch_size, patches, self.num_attention_heads, self.head_dim)
            .transpose(1, 2)
        )  # (batch, heads, seq_len, head_dim)
        value = (
            self.v_proj(hidden_states)
            .view(batch_size, patches, self.num_attention_heads, self.head_dim)
            .transpose(1, 2)
        )  # (batch, heads, seq_len, head_dim)

        # Apply RoPE if position embeddings are provided
        if position_embeddings is not None:
            cos, sin = position_embeddings
            query, key = apply_rotary_pos_emb(query, key, cos, sin)

        # Compute attention output
        scaling_factor = 1.0 / (self.head_dim**0.5)
        attn_output, attn_weights = self.attention_forward(
            query, key, value, attention_mask=attention_mask, scaling=scaling_factor
        )

        # Project back to hidden size
        attn_output = (
            attn_output.transpose(1, 2)
            .contiguous()
            .view(batch_size, patches, self.hidden_size)
        )
        attn_output = self.o_proj(attn_output)

        return attn_output, attn_weights if output_attentions else None


# ============================================================================
# Attention Layer with Residual Connection
# ============================================================================
class PixtralAttentionLayer(nn.Module):
    """
    Single layer of attention followed by feed-forward network with residual connections.
    """

    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.attention_norm = PixtralRMSNorm(config.hidden_size)
        self.ffn_norm = PixtralRMSNorm(config.hidden_size)
        self.feed_forward = PixtralMLP(config)
        self.attention = PixtralAttention(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        output_attentions: bool | None = None,
    ) -> tuple[torch.Tensor, ...]:
        """
        Args:
            hidden_states (`torch.FloatTensor`): Input to the layer of shape `(batch, seq_len, embed_dim)`.
            attention_mask (`torch.FloatTensor`): Attention mask of shape `(batch, 1, q_len, k_v_seq_len)` where padding elements are indicated by very large negative values.
            output_attentions (`bool`, *optional*, defaults to `False`): Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned tensors for more detail.
        """
        residual = hidden_states

        hidden_states = self.attention_norm(hidden_states)
        hidden_states, attn_weights = self.attention(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_embeddings=position_embeddings,
            output_attentions=output_attentions,
        )
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.ffn_norm(hidden_states)
        hidden_states = self.feed_forward(hidden_states)
        hidden_states = residual + hidden_states

        outputs = (hidden_states,)

        if output_attentions:
            outputs += (attn_weights,)

        return outputs


# ============================================================================
# Transformer consisting of multiple attention layers
# ============================================================================
class PixtralTransformer(nn.Module):
    """
    Stack of attention layers for the Pixtral model.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layers = torch.nn.ModuleList()
        for _ in range(config.num_hidden_layers):
            self.layers.append(PixtralAttentionLayer(config))

    def forward(
        self,
        input_embeds,
        attention_mask,
        position_embeddings,
        output_attentions,
        output_hidden_states,
        return_dict=False,
    ):
        """
        Args:
            input_embeds: Input tensor of shape (batch, seq_len, hidden_size)
            attention_mask: Attention mask tensor (broadcastable to (batch, heads, seq_len, seq_len))
            position_embeddings: Tuple of (cos, sin) tensors for RoPE
            output_attentions: Whether to return attention weights
            output_hidden_states: Whether to return hidden states from all layers
            return_dict: Whether to return a dictionary of outputs or a tuple
        """
        hidden_states = input_embeds
        all_attentions = (
            []
        )  # intialize it to avoid type error when output_attentions is False
        encoder_states = (
            []
        )  # intialize it to avoid type error when output_hidden_states is False

        for layer in self.layers:
            if output_hidden_states:
                encoder_states = encoder_states + [hidden_states]

            layer_outputs = layer(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                position_embeddings=position_embeddings,
                output_attentions=output_attentions,
            )

            hidden_states = layer_outputs[0]
            if output_attentions:
                all_attentions = all_attentions + [layer_outputs[1]]

        if output_hidden_states:
            encoder_states = encoder_states + [hidden_states]

        if not return_dict:
            return tuple(
                v
                for v in [hidden_states, encoder_states, all_attentions]
                if v is not None
            )

        outputs = {
            "last_hidden_state": hidden_states,
            "hidden_states": tuple(encoder_states) if output_hidden_states else None,
            "attentions": tuple(all_attentions) if output_attentions else None,
        }

        return outputs


# ============================================================================
# Vision Model for Pixtral
# ============================================================================


class PixtralVisionModel(nn.Module):
    """
    Vision model for Pixtral architecture.

    This model processes image inputs and produces hidden states that can be
    fed into a language model for vision-language tasks.
    """

    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.config = config
        self.patch_conv = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=config.hidden_size,
            kernel_size=config.patch_size,
            stride=config.patch_size,
            bias=False,
        )
        self.patch_size = config.patch_size
        self.ln_pre = PixtralRMSNorm(config.hidden_size, eps=1e-5)
        self.transformer = PixtralTransformer(config)
        self.patch_positional_embedding = PixtralRotaryEmbedding(config)

    def forward(
        self,
        pixel_values: torch.Tensor,
        image_sizes: list[tuple[int, int]] | None = None,
        output_hidden_states: bool | None = None,
        output_attentions: bool | None = None,
    ):
        """
        Args:
            pixel_values: Tensor of shape (batch, channels, height, width) containing the input images.
            image_sizes: Optional tensor of shape (batch, 2) containing the original height and width of each image in the batch. If None, it is assumed all images have the same size as the input tensor.
            output_hidden_states: Whether to return hidden states from all layers of the transformer.
            output_attentions: Whether to return attention weights from all layers of the transformer.
        """
        if image_sizes is None:
            batch_size, _, height, width = pixel_values.shape
            image_sizes = [(height, width)] * batch_size

        # pass images through initial convolution independently
        target_dtype = self.patch_conv.weight.dtype
        patch_embeds = self.patch_conv(pixel_values.to(dtype=target_dtype))
        patch_embeds_list = [
            embed[..., : (size[0] // self.patch_size), : (size[1] // self.patch_size)]
            for embed, size in zip(patch_embeds, image_sizes)
        ]

        # flatten to a single sequence
        patch_embeds = torch.cat(
            [p.flatten(1).T for p in patch_embeds_list], dim=0
        ).unsqueeze(0)
        patch_embeds = self.ln_pre(
            patch_embeds
        )  # Shape: (1, total_patches (Sum of images patches), hidden_size)

        # positional embeddings
        position_ids = (
            position_ids_in_meshgrid(
                patch_embeds_list,
                max_width=self.config.image_size // self.config.patch_size,
            )
            .unsqueeze(0)
            .to(patch_embeds.device)
        )  # Shape: (1, total_patches)

        position_embeddings = self.patch_positional_embedding(
            patch_embeds, position_ids
        )  # Tuple of (cos, sin) tensors for RoPE, each of shape (total_patches, hidden_size)

        attention_mask = generate_block_attention_mask(
            [p.shape[-2] * p.shape[-1] for p in patch_embeds_list], patch_embeds
        )

        return self.transformer(
            patch_embeds,
            attention_mask=attention_mask,
            position_embeddings=position_embeddings,
            output_hidden_states=output_hidden_states,
            output_attentions=output_attentions,
            return_dict=True,
        )  # Note Rope Added in each Transformer Layer not single time here


In [ ]:
@dataclass
class Ministral3MultimodalConfig:
    # Top-level multimodal parameters
    spatial_merge_size: int = 2
    image_token_index: int = 10
    vision_feature_layer: int = -1
    tie_word_embeddings: bool = True 

    # Nested configurations
    text_config: Ministral3Config = field(default_factory=Ministral3Config)
    vision_config: PixtralConfig = field(default_factory=PixtralConfig)

In [ ]:
class Mistral3RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (RMSNorm) for vision features.

    This module applies a rescaling-only normalization that scales the input 
    by the inverse square root of the mean of the squared hidden states. 
    It is used here to stabilize vision features before they are processed 
    by the patch merger and projection layers.

    The implementation forces calculations to `float32` for numerical stability 
    before casting back to the original input precision.

    Notes for users copying weights from Hugging Face:
    - Keep the attribute name `weight` as-is.
    - Ensure `eps` matches the `rms_norm_eps` defined in the text config.
    """

    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))
        
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """Apply RMSNorm to the input features.

        Args:
            hidden_states: Input tensor of vision features 
                (shape: [total_patches, embed_dim]).

        Returns:
            torch.Tensor: Normalized and rescaled features with the same 
                shape as the input.
        """
        org_input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.eps)
        return self.weight * hidden_states.to(org_input_dtype)


class Mistral3PatchMerger(nn.Module):
    """Learned merging of small spatial patches into larger patches.

    The module collects groups of `spatial_merge_size**2` neighboring patches
    and applies a single linear layer (`merging_layer`) to reduce the
    concatenated embeddings back to `hidden_size`.

    Notes for users copying weights from Hugging Face:
    - Keep the attribute name `merging_layer` as-is (it maps to checkpoint keys).
    - Do not change `spatial_merge_size` or `patch_size` semantics here.
    """

    def __init__(self, config: Ministral3MultimodalConfig) -> None:
        super().__init__()

        hidden_size = config.vision_config.hidden_size
        self.spatial_merge_size = config.spatial_merge_size
        self.patch_size = config.vision_config.patch_size

        # A learned linear projection that reduces concatenated patch vectors
        # (embed_dim * merge_size**2) -> hidden_size. Keep name for checkpointing.
        self.merging_layer = nn.Linear(
            hidden_size * self.spatial_merge_size**2, hidden_size, bias=False
        )

    def forward(
        self, image_features: torch.Tensor, image_sizes: Iterable[torch.Tensor]
    ) -> torch.Tensor:
        """Merge image patch tokens into larger patches.

        Args:
            image_features: concatenated patch tokens for all images in batch
                (shape: [total_patches, embed_dim]). The tensor is expected to
                be a concatenation of per-image token sequences.
            image_sizes: iterable of per-image sizes used to reshape tokens.
                Each entry should be convertible to (H_patches, W_patches)
                so that H_patches * W_patches equals the number of tokens for
                that image.

        Returns:
            Tensor with merged patches projected to `hidden_size`.
        """

        # Compute how many patch tokens each image contributes (in patch-grid units)
        patch_grid_sizes = [
            (
                int(image_size[0]) // self.patch_size,
                int(image_size[1]) // self.patch_size,
            )
            for image_size in image_sizes
        ]

        tokens_per_image = [h * w for h, w in patch_grid_sizes]
        embed_dim = image_features.shape[-1]

        # We'll accumulate per-image merged patch blocks and concatenate later.
        permuted_tensor = []

        # Split the flat image_features into per-image token tensors
        for image_index, image_tokens in enumerate(
            image_features.split(tokens_per_image)
        ):
            # Reconstruct the 2D grid of patches for this image
            # `image_sizes` may contain pixel dims, but we already converted
            # them into patch-grid dims above; reuse those dims here to reshape.
            h_patches, w_patches = patch_grid_sizes[image_index]

            # image_tokens: (H_patches * W_patches, embed_dim)
            # reshape -> (H_patches, W_patches, embed_dim)
            image_grid = image_tokens.view(h_patches, w_patches, embed_dim)

            # Move channel/embed dim to front and add batch dim for unfold:
            # (1, embed_dim, H_patches, W_patches)
            image_grid = image_grid.permute(2, 0, 1).unsqueeze(0)

            # Use unfold to extract non-overlapping blocks of size spatial_merge_size
            grid = torch.nn.functional.unfold(
                image_grid,
                kernel_size=self.spatial_merge_size,
                stride=self.spatial_merge_size,
            )

            # grid shape -> (1, embed_dim * merge_size**2, N_windows)
            # Reshape to (N_windows, embed_dim * merge_size**2) so we can apply
            # the linear merging layer across the last dim.
            grid = grid.view(embed_dim * self.spatial_merge_size**2, -1).t()

            permuted_tensor.append(grid)

        # Concatenate merged blocks for all images and apply the learned linear layer
        merged_patches = torch.cat(permuted_tensor, dim=0)
        merged_patches = self.merging_layer(merged_patches)
        return merged_patches


class Mistral3MultiModalProjector(nn.Module):
    """Project vision tokens into Mistral3 text embedding space.

    This thin module performs the following steps in order:
    1. RMS normalization (`norm`) on incoming vision features.
    2. Merge neighboring patches with `patch_merger`.
    3. Two linear layers with GELU non-linearity (`linear_1`, `act`, `linear_2`)
       to map vision `hidden_size` -> text `hidden_size`.

    All layer attribute names are preserved to allow direct weight loading
    from checkpoints.
    """

    def __init__(self, config: Ministral3MultimodalConfig) -> None:
        super().__init__()

        # Normalize vision features before merging/projection
        self.norm = Mistral3RMSNorm(
            config.vision_config.hidden_size, eps=config.text_config.rms_norm_eps
        )

        # Patch merger keeps its attribute name to match checkpoints
        self.patch_merger = Mistral3PatchMerger(config)

        # Two projection layers with a GELU activation in between.
        # Keep layer names `linear_1` and `linear_2` for checkpoint compatibility.
        self.linear_1 = nn.Linear(
            config.vision_config.hidden_size, config.text_config.hidden_size, bias=False
        )
        self.act = nn.GELU()
        self.linear_2 = nn.Linear(
            config.text_config.hidden_size, config.text_config.hidden_size, bias=False
        )

    def forward(
        self, image_features: torch.Tensor, image_sizes: Iterable[torch.Tensor]
    ) -> torch.Tensor:
        """Forward pass: normalize, merge, and project image patches.

        Args:
            image_features: concatenated patch tokens for the batch
                (shape: [total_patches, vision_hidden_size]).
            image_sizes: iterable with per-image sizes used by the patch merger.

        Returns:
            Tensor: projected patches in text embedding space
                (shape: [total_merged_patches, text_hidden_size]).
        """

        # RMS normalize incoming features
        normed_patches = self.norm(image_features)

        # Merge spatial patches into larger patches
        merged_patches = self.patch_merger(normed_patches, image_sizes)

        # Project merged vision patches into text hidden space
        projected_patches = self.linear_1(merged_patches)
        projected_patches = self.act(projected_patches)
        projected_patches = self.linear_2(projected_patches)

        return projected_patches


In [ ]:
# -----------------------
# Final Multimodal Wrapper
# -----------------------

@dataclass
class Ministral3MultimodalConfig:
    spatial_merge_size: int = 2
    image_token_index: int = 10
    vision_feature_layer: int = -1
    tie_word_embeddings: bool = True 
    text_config: Ministral3Config = field(default_factory=Ministral3Config)
    vision_config: PixtralConfig = field(default_factory=PixtralConfig)

class Ministral3MultimodalModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.vision_tower = PixtralVisionModel(config.vision_config)
        self.multi_modal_projector = Mistral3MultiModalProjector(config)
        self.language_model = Ministral3Model(config.text_config)

    def _replace_image_tokens(self, input_ids, inputs_embeds, image_features):
        # 1. Create mask where image tokens are located
        image_token_mask = (input_ids == self.config.image_token_index)
        
        # 2. Expand mask to match embedding dimensions
        expanded_mask = image_token_mask.unsqueeze(-1).expand_as(inputs_embeds)
        
        # 3. SAFETY FIX: Ensure vision features are contiguous in memory
        image_features = image_features.to(device=inputs_embeds.device, dtype=inputs_embeds.dtype).contiguous()
        
        # 4. Inject vision features into text embeddings
        return inputs_embeds.masked_scatter(expanded_mask, image_features)

    def forward(self, input_ids, pixel_values=None, image_sizes=None, attention_mask=None, past_key_values=None, inputs_embeds=None, cache_position=None):
        if inputs_embeds is None:
            inputs_embeds = self.language_model.embed_tokens(input_ids)

        if pixel_values is not None:
            # Fake vision forward for this snippet (In real code, call self.vision_tower)
            # image_features = self.vision_tower(pixel_values, image_sizes)
            # projected_vision = self.multi_modal_projector(image_features)
            
            # For demonstration, we assume projected_vision is ready
            # inputs_embeds = self._replace_image_tokens(input_ids, inputs_embeds, projected_vision)
            pass 

        return self.language_model(
            input_ids=None,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            cache_position=cache_position
        )

class Ministral3ForConditionalGeneration(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.model = Ministral3MultimodalModel(config)
        self.lm_head = nn.Linear(config.text_config.hidden_size, config.text_config.vocab_size, bias=False)
        
        # Tie weights
        self.lm_head.weight = self.model.language_model.embed_tokens.weight

    def forward(self, input_ids=None, pixel_values=None, logits_to_keep=0, **kwargs):
        outputs = self.model(input_ids=input_ids, pixel_values=pixel_values, **kwargs)
        hidden_states = outputs["last_hidden_state"]

        # Optimization: Only compute logits for the requested tokens
        if logits_to_keep > 0:
            hidden_states = hidden_states[:, -logits_to_keep:, :]

        logits = self.lm_head(hidden_states)
        return {"logits": logits, "past_key_values": outputs["past_key_values"]}

## 3. Weight Loading, Inference & UI

This section handles the model lifecycle:
1. **Zero-RAM Initialization**: We construct the model directly on the GPU using `with torch.device("cuda"):`. This prevents the system from allocating 6GB of CPU RAM for initialization + 6GB of VRAM for moving it, keeping CPU RAM free for data processing.
2. **Robust Loading**: A custom loader maps Hugging Face checkpoint keys (`language_model.model...`) to our custom class structure (`model.language_model...`).
3. **Optimized Inference**: The `generate` function uses KV-Caching and calculates logits *only* for the last token, ensuring high speed and low memory usage.

In [ ]:
# -----------------------
# Weight Loading Utilities
# -----------------------
from safetensors import safe_open
from huggingface_hub import snapshot_download

HF_REPO = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
LOCAL_DIR = "./saved_model/Ministral3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

def remap_key(key):
    """Maps HF checkpoint keys to our custom model structure."""
    if key.startswith("language_model.model."):
        return key.replace("language_model.model.", "model.language_model.")
    if key.startswith("vision_tower."):
        return f"model.{key}"
    if key.startswith("multi_modal_projector."):
        return f"model.{key}"
    if key.startswith("language_model.lm_head."):
        return key.replace("language_model.lm_head.", "lm_head.")
    if not key.startswith("model.") and not key.startswith("lm_head."):
         return f"model.{key}"
    return key

def _set_nested_param(model, key, tensor):
    try:
        module_name, param_name = key.rsplit(".", 1) if "." in key else ("", key)
        submodule = model.get_submodule(module_name) if module_name else model
        param = getattr(submodule, param_name)
        
        if param.shape != tensor.shape and param.numel() == tensor.numel():
            tensor = tensor.view(param.shape)
            
        with torch.no_grad():
            param.data = tensor
    except Exception as e:
        print(f"Warning: Failed to load {key}: {e}")

def load_weights_into_model(model, directory, device):
    from pathlib import Path
    files = list(Path(directory).glob("*.safetensors"))
    print(f"Loading weights from {len(files)} files...")
    
    for file in files:
        with safe_open(file, framework="pt", device=str(device)) as f:
            for key in f.keys():
                custom_key = remap_key(key)
                tensor = f.get_tensor(key)
                _set_nested_param(model, custom_key, tensor)
        torch.cuda.empty_cache()
    print("Weights loaded successfully.")

def get_model_and_processor():
    print(f"Downloading {HF_REPO}...")
    snapshot_download(repo_id=HF_REPO, local_dir=LOCAL_DIR, allow_patterns=["*.safetensors", "*.json"])

    print("Initializing Model on GPU...")
    
    # Configuration matches 3B Instruct
    rope_params = RopeParameters(rope_type="yarn", original_max_position_embeddings=16384)
    text_config = Ministral3Config(
        hidden_size=3072, intermediate_size=9216, num_hidden_layers=26, 
        num_attention_heads=32, num_key_value_heads=8, head_dim=128,
        vocab_size=131072, tie_word_embeddings=True, rope_parameters=rope_params.__dict__
    )
    vision_config = PixtralConfig(hidden_size=1024, head_dim=64, num_attention_heads=16)
    
    multimodal_config = Ministral3MultimodalConfig(
        spatial_merge_size=2, image_token_index=10, 
        text_config=text_config, vision_config=vision_config
    )

    # CRITICAL: Initialize directly on GPU to save CPU RAM
    torch.set_default_dtype(DTYPE)
    with torch.device(DEVICE):
        model = Ministral3ForConditionalGeneration(multimodal_config)
    torch.set_default_dtype(torch.float32)

    load_weights_into_model(model, LOCAL_DIR, DEVICE)
    model.eval()
    
    # CRITICAL FIX: Load processor from Repo ID, not model object
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(HF_REPO)
    
    return model, processor

In [ ]:
# -----------------------
# Inference Generation
# -----------------------

@torch.no_grad()
def generate(model, processor, image, prompt, max_tokens=256, temp=0.7):
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    
    # Apply chat template
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(images=image, text=text_prompt, return_tensors="pt")
    
    input_ids = inputs["input_ids"].to(DEVICE)
    pixel_values = inputs["pixel_values"].to(DEVICE, dtype=DTYPE)
    
    # Extract attention mask if present
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    # Image sizes handling
    image_sizes = inputs.get("image_sizes")
    if image_sizes is not None:
        image_sizes = image_sizes.to(DEVICE)

    generated_ids = []

    # --- 1. Prefill Step ---
    # We pass logits_to_keep=1 to only compute the logit for the last token.
    # This prevents OOM by not materializing (Batch, Seq_Len, Vocab) tensor.
    outputs = model(
        input_ids=input_ids,
        pixel_values=pixel_values,
        image_sizes=image_sizes,
        attention_mask=attention_mask,
        past_key_values=None,
        logits_to_keep=1 
    )
    
    next_logits = outputs["logits"][:, -1, :] # Logits are already sliced to size 1
    kv_cache = outputs["past_key_values"]

    # --- 2. Decode Loop ---
    for _ in range(max_tokens):
        if temp > 0:
            probs = torch.softmax(next_logits / temp, dim=-1)
            next_token = torch.multinomial(probs, 1)
        else:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)

        token_id = next_token.item()
        if token_id == processor.tokenizer.eos_token_id:
            break
            
        generated_ids.append(token_id)
        
        # Update attention mask for the new token (simple concatenation for batch=1)
        if attention_mask is not None:
            attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=DEVICE)], dim=1)

        # Forward pass with cached keys
        outputs = model(
            input_ids=next_token,
            pixel_values=None, # Vision not needed in decode
            attention_mask=attention_mask,
            past_key_values=kv_cache,
            logits_to_keep=1
        )
        
        next_logits = outputs["logits"][:, -1, :]
        kv_cache = outputs["past_key_values"]

    return processor.tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
# -----------------------
# Main Execution & UI
# -----------------------
import gradio as gr

# Load Model once
model, processor = get_model_and_processor()

def run_inference(image, text, max_new, temp):
    if image is None: 
        return "Please upload an image."
    if not text: 
        text = "Describe this image."
    
    try:
        return generate(model, processor, image, text, int(max_new), float(temp))
    except Exception as e:
        import traceback
        return f"Error: {str(e)}\n{traceback.format_exc()}"

with gr.Blocks(title="Ministral-3B Multimodal (Kaggle)") as app:
    gr.Markdown("### Ministral-3 (3B) Multimodal - Zero RAM Init")
    
    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="Input Image")
            txt_input = gr.Textbox(label="Prompt", value="Describe this image in detail.")
            
            with gr.Accordion("Parameters", open=False):
                max_tokens = gr.Slider(64, 1024, 256, label="Max Tokens")
                temperature = gr.Slider(0.0, 1.0, 0.7, label="Temperature")
            
            btn = gr.Button("Generate", variant="primary")
        
        with gr.Column():
            output = gr.Markdown(label="Response")

    btn.click(run_inference, [img_input, txt_input, max_tokens, temperature], output)

# Launch with share=True for public link
app.launch(share=True, debug=True)